# Trustworthy and Resilient Network Intrusion Detection

**A reproducible proof-of-execution pilot**  
Wisam Makki Salim

This notebook orchestrates the audited pipeline. It does not claim a new classifier or a completed study. The emphasis is leakage control, operational trade-offs, and reproducibility.

## 1. Data and provenance

Download `Merged01.csv` from the official CICIoT2023 portal linked by the University of New Brunswick and place it at `data/ciciot2023/Merged01.csv`. The notebook verifies the locked SHA-256 before training. Raw data are not redistributed.

Official source: https://www.unb.ca/cic/datasets/iotdataset-2023.html  
Article: https://doi.org/10.3390/s23135941

In [ ]:
from pathlib import Path
import hashlib

ROOT = Path.cwd()
DATA = ROOT / 'data' / 'ciciot2023' / 'Merged01.csv'
EXPECTED = '8b43d6552a8cafd3b0ca2cedf6464ca3fe644d7fc9bb5dfe906368f1542792fe'
assert DATA.exists(), f'Missing official dataset shard: {DATA}'
h = hashlib.sha256(DATA.read_bytes()).hexdigest()
assert h == EXPECTED, f'Unexpected input hash: {h}'
print('Verified:', DATA, h)

## 2. G1 - Dataset audit

This stage verifies schema, label balance, missingness, infinite values, exact duplicates, feature duplicates, and conflicting binary labels.

In [ ]:
%run scripts/g1_audit.py
import json
g1 = json.load(open('outputs/g1/g1_audit.json'))
{k: g1[k] for k in ['shape', 'binary_counts', 'missing_cells', 'infinite_numeric_cells', 'exact_duplicate_rows', 'exact_duplicate_feature_rows', 'feature_vectors_with_conflicting_binary_labels']}

## 3. G2 - Leakage-controlled baselines

Logistic Regression and Random Forest use development-only preprocessing, class weighting, and a default threshold of 0.50. Identical feature vectors cannot cross partitions.

In [ ]:
%run scripts/g2_baselines.py
import pandas as pd
pd.read_csv('outputs/g2/g2_results_table.csv')

## 4. G3 - Validation-only operating-point selection

The locked rule maximizes attack recall subject to validation FPR <= 1%. If recall differs by no more than 0.1 percentage points, the smaller serialized model is preferred. The rule is frozen before applying it to test predictions.

In [ ]:
%run scripts/g3_pareto_selection.py
from IPython.display import Image, display
display(Image('outputs/g3/g3_pareto_tradeoff.png'))
pd.read_csv('outputs/g3/g3_selected_results_table.csv')

## 5. G4 - Uncertainty and stability

The frozen decision is evaluated with 10,000 stratified bootstrap replicates. No threshold is retuned after test results are visible.

In [ ]:
%run scripts/g4_uncertainty.py
display(Image('outputs/g4/g4_uncertainty_stability.png'))
pd.read_csv('outputs/g4/g4_confidence_intervals.csv')

## 6. Interpretation

**Implemented:** official-data verification, leakage audit, reproducible baselines, validation-only constrained selection, frozen test evaluation, computational trade-offs, and uncertainty analysis.

**Observed:** the selected Random Forest achieved 98.167% attack recall and 1.208% FPR on the held-out split; the 95% FPR interval was 0.846% to 1.601%.

**Inferred:** point-estimate constraint satisfaction on validation is not a reliable operational guarantee.

**Proposed next step:** confidence-bounded adaptive thresholding, external validation, and temporal/device-group evaluation under distribution shift.